# IngestService Step-by-Step Testing

This notebook tests each method of `IngestService` individually to help debug and understand the ingestion pipeline.

## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path
%load_ext autoreload
%autoreload 2
# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

In [ ]:
from src.ingest import IngestService
from dotenv import load_dotenv

load_dotenv()

print("Imports successful!")

## 2. Initialize IngestService

In [ ]:
# Initialize the service
service = IngestService(
    weaviate_host="localhost",
    weaviate_port=8080,
    grpc_port=50051,
    base_dir=project_root
)

print(f"Base directory: {service.base_dir}")
print(f"Weaviate host: {service.weaviate_host}:{service.weaviate_port}")

## 3. Test Weaviate Connection

In [ ]:
# Test connection to Weaviate
try:
    client = service._get_client()
    print(f"Connected to Weaviate: {client.is_ready()}")
    
    # List existing collections
    collections = client.collections.list_all()
    print(f"\nExisting collections: {list(collections.keys())}")
except Exception as e:
    print(f"Connection failed: {e}")
    print("\nMake sure Weaviate is running (docker-compose up -d)")

## 4. Test load_document()

Load and parse a PDF file into document elements.

In [ ]:
# List available files in data/raw
raw_data_path = project_root / "data" / "raw"
available_files = list(raw_data_path.glob("*"))

print("Available files in data/raw:")
for f in available_files:
    print(f"  - {f.name}")

In [ ]:
# Set the file to test with
TEST_FILE = "RAG university.pdf"  # Change this to your test file

print(f"Testing with file: {TEST_FILE}")

In [ ]:
# Load the document
try:
    documents = service.preprocess_documents(TEST_FILE)
    print(f"Loaded {len(documents)} documents")
    
except Exception as e:
    print(f"Error loading document: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Explore document types
if documents:
    from collections import Counter
    
    types = [doc.metadata.get('type', 'unknown') for doc in documents]
    type_counts = Counter(types)
    
    print("Document type distribution:")
    for doc_type, count in type_counts.items():
        print(f"  {doc_type}: {count}")

In [ ]:
for doc in documents:
    if doc.metadata.get('type', '') == 'Table':
        print(doc)

## 5. Test create_collection()

In [ ]:
# Create a test collection
TEST_COLLECTION = "TestCollection"
service.create_collection(TEST_COLLECTION)
   

## 6. Test ingest_chunks()

In [ ]:
# Ingest the loaded documents (if available)
if documents:
    try:
        # Limit to first 5 documents for testing
        test_docs = documents
        print(f"Ingesting {len(test_docs)} test documents...")
        
        result = service.add_documents(TEST_COLLECTION, test_docs)
        print(f"\nIngest result: {result}")
    except Exception as e:
        print(f"Error ingesting chunks: {e}")
        import traceback
        traceback.print_exc()
else:
    print("No documents to ingest. Load documents first (Step 4).")

In [ ]:
service.ingest(
    file_name=TEST_FILE,
    collection_name=TEST_COLLECTION
)

In [ ]:
client = service._get_client()
collection = client.collections.get(TEST_COLLECTION)

results = collection.query.fetch_objects(
    limit=5,
    include_vector=True  # This includes the embedding vector
)
for i, obj in enumerate(results.objects, 1):
    props = obj.properties
    vector = obj.vector.get('default', [])  # Get the default vector
    print(f"\n[{i}] UUID: {obj.uuid}")
    print(f"    Type: {props.get('type', 'N/A')}")
    print(f"    Text: {props.get('text', '')[:100]}...")
    print(f"    Vector dimension: {len(vector)}")
    print(f"    Vector (first 10 values): {vector[:10]}")

## 7. Test delete_collection()

In [ ]:
# Delete the test collection
service._get_client()
service.delete_collection(TEST_COLLECTION)


In [ ]:
from src.retriever import retrieve

In [ ]:
objects = retrieve(
    query="what average score of 'I needed to learn a lot of things before I could get going with this system.' ?",
    collection_name=TEST_COLLECTION,
    top_k=10,
    top_k_reranker=3
)

In [ ]:
from IPython.display import display, Image

for i, obj in enumerate(objects, 1):
    props = obj.properties
    obj_type = props.get('type', '')
    
    print(f"\n{'='*60}")
    print(f"[{i}] Type: {obj_type} | Source: {props.get('source', 'N/A')}")
    print(f"{'='*60}")
    
    if obj_type in ['Image', 'Table']:
        image_path = props.get('image_path', '')
        if image_path:
            display(Image(filename=image_path, width=500))
        else:
            print("No image path available")
    else:
        print(props.get('text', 'No text available'))

In [ ]:
import weaviate

# Connect to local Weaviate
client = weaviate.connect_to_local(
    host="localhost",
    port=8080,
    grpc_port=50051
)

try:
    collection = client.collections.get("NewChunkCollection")

    # Fetch all objects, only returning the 'source' property
    results = collection.query.fetch_objects(
        limit=200,
        return_properties=["source"]
    )

    # Collect unique sources and show a sample
    sources = []
    for obj in results.objects:
        sources.append(obj.properties.get("source", ""))

    unique_sources = sorted(set(sources))

    print(f"Total objects fetched: {len(sources)}")
    print(f"Unique 'source' values: {len(unique_sources)}")
    print("-" * 60)
    for s in unique_sources:
        count = sources.count(s)
        print(f"  [{count:3d} chunks] {s}")

    # Show a few full object samples for detail
    print("\n" + "=" * 60)
    print("Sample objects (first 5):")
    print("=" * 60)
    for i, obj in enumerate(results.objects[:5]):
        print(f"\n--- Object {i+1} ---")
        print(f"  UUID:   {obj.uuid}")
        print(f"  source: {obj.properties.get('source', '')}")

finally:
    client.close()
